In [9]:
%load_ext autoreload
%autoreload 2
import sys, time, itertools
import numpy as np, pandas as pd, lightgbm as lgb
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name in ("notebooks", "futuro") else Path.cwd()
for p in ("src", "cv", "futuro", "futuro/pruebas"):
    if str(ROOT / p) not in sys.path:
        sys.path.insert(0, str(ROOT / p))

import m_lgb_future
import features_future as ff
from metric import hits, labels, report

# 1. Cargamos los datos con el mejor bloque probado hasta ahora (+clear, sin coord antiguo que metía ruido)
print("Cargando base + logic + past + future + clear...")
data = m_lgb_future.build_data()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Cargando base + logic + past + future + clear...
474688 filas etiquetadas, 66409 de test; bloques: base=88, logic=14, future=12, past=11, coord=12, clear=10  (36s)


In [11]:
TYPES = [
    'entry_conflict_resolution', 'exit_conflict_resolution', 'entry_coordination', 'exit_coordination',
    'assume_aircraft', 'transfer_aircraft', 'conflict_resolution', 'non_conformance_resolution',
    'qos_improvement', 'entry_conditions', 'exit_conditions', 'aircraft_request', 'zone_conflict',
    'return_to_route'
]
def add_ultra_future_features(df_full):
    """
    Se calcula por sesión ordenando por timestamp, sin usar jamás target_index.
    """
    df = df_full.sort_values(["session", "timestamp", "row_index"]).reset_index(drop=True).copy()
    
    # CONVERSIÓN CRUCIAL: Aseguramos que timestamp sea numérico para poder restar tiempos
    if not pd.api.types.is_numeric_dtype(df["timestamp"]):
        df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")

    # Extraer callsign del avión (si no está ya creado)
    if "cs" not in df.columns:
        pref = df.task_index.map(lambda t: len(TYPES[t]) + 1)
        df["cs"] = [n[p:].split("_")[0] for n, p in zip(df.task_name, pref)]

    out = pd.DataFrame(index=df.index)

    # -------------------------------------------------------------------------
    # A. MICRO-DELTAS EN LA SESIÓN POR TAREA (session + task_name)
    # -------------------------------------------------------------------------
    g_task = df.groupby(["session", "task_name"], sort=False)
    
    out["uf_dt_next_task"] = g_task["timestamp"].shift(-1) - df["timestamp"]
    out["uf_dt_prev_task"] = df["timestamp"] - g_task["timestamp"].shift(1)
    
    # Si la tarea desaparece en el siguiente snapshot de la sesión
    max_ts_session = df.groupby("session")["timestamp"].transform("max")
    next_ts_session = df[["session", "timestamp"]].drop_duplicates().sort_values(["session", "timestamp"])
    next_ts_session["next_snap_ts"] = next_ts_session.groupby("session")["timestamp"].shift(-1)
    df = df.merge(next_ts_session, on=["session", "timestamp"], how="left")
    
    next_task_ts = g_task["timestamp"].shift(-1)
    out["uf_vanishes_next_snap"] = (
        (df["timestamp"] < max_ts_session) & 
        (next_task_ts.isna() | (next_task_ts > df["next_snap_ts"]))
    ).astype("float32")
    
    next2_task_ts = g_task["timestamp"].shift(-2)
    out["uf_steps_left_in_block"] = (
        next_task_ts.notna().astype("int8") + next2_task_ts.notna().astype("int8")
    ).astype("float32")

    # Reseteo o caída de x_state_age en el futuro inmediato
    if "x_state_age" in df.columns:
        next_state_age = g_task["x_state_age"].shift(-1)
        out["uf_state_age_reset_next"] = (
            next_state_age.notna() & (next_state_age < df["x_state_age"])
        ).astype("float32")
        out["uf_state_age_delta_prev"] = (df["x_state_age"] - g_task["x_state_age"].shift(1)).astype("float32")

    # -------------------------------------------------------------------------
    # B. MICRO-CAMBIOS DE COORDINACIÓN Y HERRAMIENTAS POR AVIÓN (session + cs)
    # -------------------------------------------------------------------------
    ac_cols = [
        "is_assumed", "x_transfer_in", "x_isc_notified", "x_isc_coordinated",
        "k_out_NOTIFIED", "k_out_COORDINATED", "k_outCoordMismatch",
        "k_coordinationToolActivation", "k_transferToolActivation",
        "k_aircraftNeedsToBeTransferred", "k_exitConditionsAreMet",
        "k_outCoordRevision", "a_since_flight"
    ]
    ac_cols = [c for c in ac_cols if c in df.columns]
    
    ac_snap = df[["session", "cs", "timestamp"] + ac_cols].drop_duplicates(["session", "cs", "timestamp"]).sort_values(["session", "cs", "timestamp"])
    g_ac_time = ac_snap.groupby(["session", "cs"], sort=False)
    
    if "a_since_flight" in ac_snap.columns:
        asf_next1 = g_ac_time["a_since_flight"].shift(-1)
        asf_next2 = g_ac_time["a_since_flight"].shift(-2)
        ac_snap["ac_acted_next1"] = (asf_next1 < ac_snap["a_since_flight"]).astype("float32")
        ac_snap["ac_acted_next2"] = (asf_next2 < asf_next1).astype("float32")
        ac_snap["ac_min_asf_next2"] = np.minimum(asf_next1.fillna(9999), asf_next2.fillna(9999)).astype("float32")

    if "is_assumed" in ac_snap.columns:
        ac_snap["ac_assumed_next1"] = ((ac_snap["is_assumed"] == 0) & (g_ac_time["is_assumed"].shift(-1) == 1)).astype("float32")
        ac_snap["ac_transferred_next1"] = ((ac_snap["is_assumed"] == 1) & (g_ac_time["is_assumed"].shift(-1) == 0)).astype("float32")
    
    if "x_isc_coordinated" in ac_snap.columns:
        ac_snap["ac_in_coord_change_next1"] = (ac_snap["x_isc_coordinated"].fillna(-1) != g_ac_time["x_isc_coordinated"].shift(-1).fillna(-1)).astype("float32")
        ac_snap["ac_in_coord_change_next2"] = (ac_snap["x_isc_coordinated"].fillna(-1) != g_ac_time["x_isc_coordinated"].shift(-2).fillna(-1)).astype("float32")

    if "k_out_COORDINATED" in ac_snap.columns:
        ac_snap["ac_out_coord_change_next1"] = (ac_snap["k_out_COORDINATED"].fillna(-1) != g_ac_time["k_out_COORDINATED"].shift(-1).fillna(-1)).astype("float32")
        ac_snap["ac_out_coord_change_next2"] = (ac_snap["k_out_COORDINATED"].fillna(-1) != g_ac_time["k_out_COORDINATED"].shift(-2).fillna(-1)).astype("float32")

    if "k_coordinationToolActivation" in ac_snap.columns:
        ac_snap["ac_coord_tool_next1"] = (g_ac_time["k_coordinationToolActivation"].shift(-1) == 1).astype("float32")
        ac_snap["ac_coord_tool_prev1"] = (g_ac_time["k_coordinationToolActivation"].shift(1) == 1).astype("float32")

    if "k_outCoordRevision" in ac_snap.columns:
        ac_snap["ac_rev_inc_next1"] = (g_ac_time["k_outCoordRevision"].shift(-1) > ac_snap["k_outCoordRevision"]).astype("float32")

    new_ac_cols = [c for c in ac_snap.columns if c.startswith("ac_")]
    df = df.merge(ac_snap[["session", "cs", "timestamp"] + new_ac_cols], on=["session", "cs", "timestamp"], how="left")
    for c in new_ac_cols:
        out["uf_" + c] = df[c]

    # -------------------------------------------------------------------------
    # C. INTERACCIÓN TIPO DE TAREA × EVENTO FUTURO ESPECÍFICO DEL AVIÓN
    # -------------------------------------------------------------------------
    t = df["task_index"]
    out["uf_match_assume_next1"] = ((t == 4) & (df.get("ac_assumed_next1", 0) == 1)).astype("float32")
    out["uf_match_entry_coord_next"] = (
        (t == 2) & ((df.get("ac_in_coord_change_next1", 0) == 1) | (df.get("ac_in_coord_change_next2", 0) == 1)) & (df.get("ac_assumed_next1", 0) == 0)
    ).astype("float32")
    
    out["uf_match_transfer_next1"] = ((t == 5) & (df.get("ac_transferred_next1", 0) == 1)).astype("float32")
    out["uf_match_exit_coord_next"] = (
        (t == 3) & (
            (df.get("ac_out_coord_change_next1", 0) == 1) | 
            (df.get("ac_out_coord_change_next2", 0) == 1) | 
            (df.get("ac_coord_tool_next1", 0) == 1) | 
            (df.get("ac_rev_inc_next1", 0) == 1)
        )
    ).astype("float32")

    # -------------------------------------------------------------------------
    # D. COMPETICIÓN INTRA-AVIÓN Y INTRA-SNAPSHOT DE FUTURO
    # -------------------------------------------------------------------------
    df["_vanish_next"] = out["uf_vanishes_next_snap"]
    g_snap_ac = df.groupby(["sample_id", "cs"], sort=False)
    
    out["uf_ac_sum_vanish_next"] = g_snap_ac["_vanish_next"].transform("sum").astype("float32")
    out["uf_is_only_ac_task_vanishing"] = ((out["uf_vanishes_next_snap"] == 1) & (out["uf_ac_sum_vanish_next"] == 1)).astype("float32")

    end_cols = [c for c in df.columns if "end" in c or "next_input" in c or c.startswith("f_") or c.startswith("sx_")]
    for c in end_cols[:8]:
        if pd.api.types.is_numeric_dtype(df[c]):
            out[f"uf_ac_diff_{c}"] = (df[c] - g_snap_ac[c].transform("min")).astype("float32")
            out[f"uf_snap_rank_{c}"] = df.groupby("sample_id")[c].rank(method="min", pct=True).astype("float32")

    out["uf_snap_total_vanishing_next"] = df.groupby("sample_id")["_vanish_next"].transform("sum").astype("float32")

    return out.fillna(0.0)

In [12]:
# ==========================================
# CELDA 3: Carga unificada de datos y features
# ==========================================
import time
import numpy as np
import pandas as pd

META = {'sample_id', 'target_index', 'session', 'atco', 'scenario', 'timestamp', 
        'task_name', 'row_index', 'fold', 'split', 'cs', 'block', 'blk_pos', 't', 'y', 'type'}

# 1. Obtenemos el DataFrame principal de forma robusta según lo que devuelva build_data()
if isinstance(data, dict):
    # Buscamos DataFrames dentro del diccionario
    dfs = [v for v in data.values() if isinstance(v, pd.DataFrame)]
    if len(dfs) == 1:
        df_all = dfs[0].copy()
    elif len(dfs) >= 2:
        df_all = pd.concat(dfs, ignore_index=True)
    else:
        # Si build_data devolvió otra estructura interna, probamos llamando directamente a la función base
        df_all = ff.add_folds(ff.load())
        
    # Extraemos las features base disponibles
    feat_lists = [v for v in data.values() if isinstance(v, (list, tuple, pd.Index)) and len(v) > 10 and isinstance(v[0], str)]
    if feat_lists:
        base_feat_cols = [c for c in feat_lists[0] if c in df_all.columns and c not in META]
    else:
        base_feat_cols = [c for c in df_all.columns if c not in META and pd.api.types.is_numeric_dtype(df_all[c])]
elif isinstance(data, (tuple, list)):
    dfs = [x for x in data if isinstance(x, pd.DataFrame)]
    df_all = pd.concat(dfs, ignore_index=True) if len(dfs) >= 2 else dfs[0].copy()
    base_feat_cols = [c for c in df_all.columns if c not in META and pd.api.types.is_numeric_dtype(df_all[c])]
else:
    df_all = data.copy()
    base_feat_cols = [c for c in df_all.columns if c not in META and pd.api.types.is_numeric_dtype(df_all[c])]

if "split" not in df_all.columns:
    df_all["split"] = np.where(df_all["target_index"].notna(), "train", "test")

base_feat_cols = list(dict.fromkeys(base_feat_cols))
print(f"Base cargada: {len(df_all):,} filas | {len(base_feat_cols)} features base.")

# 2. Calculamos las nuevas features ultra-futuro y de competición intra-avión
print(f"Calculando nuevas features uf_*...")
t0 = time.time()
uf_df = add_ultra_future_features(df_all)

# Alineamos rigurosamente los índices ordenando igual
df_all = df_all.sort_values(["session", "timestamp", "row_index"]).reset_index(drop=True)
uf_df = uf_df.reset_index(drop=True)
df_all = pd.concat([df_all, uf_df], axis=1)

ALL_FEATS = list(dict.fromkeys(base_feat_cols + list(uf_df.columns)))
print(f"¡Listo en {time.time() - t0:.1f}s! -> Total features combinadas: {len(ALL_FEATS)} ({len(uf_df.columns)} nuevas).")

lab_df = df_all[df_all["split"] != "test"].sort_values(["sample_id", "row_index"]).reset_index(drop=True)
test_df = df_all[df_all["split"] == "test"].sort_values(["sample_id", "row_index"]).reset_index(drop=True)

# 3. Definición de modelos complementarios (PARAMS_A y PARAMS_B) y función de entrenamiento
PARAMS_A = dict(
    objective="lambdarank", n_estimators=450, learning_rate=0.045,
    num_leaves=31, min_child_samples=50, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.65, lambdarank_truncation_level=5, reg_lambda=1.0, verbose=-1, n_jobs=-1
)

PARAMS_B = dict(
    objective="lambdarank", n_estimators=650, learning_rate=0.03,
    num_leaves=45, min_child_samples=90, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.45, lambdarank_truncation_level=3, reg_lambda=5.0, verbose=-1, n_jobs=-1
)

def train_oof_and_test(lab, test, feats, params_list, seeds=(0, 1, 2)):
    oof_preds = np.zeros(len(lab), dtype="float64")
    test_preds = np.zeros(len(test), dtype="float64")
    total_models = len(params_list) * len(seeds)
    
    for f in range(5):
        t_f = time.time()
        tr_mask = (lab["fold"] != f).values
        va_mask = (lab["fold"] == f).values
        tr_data = lab.loc[tr_mask]
        va_data = lab.loc[va_mask]
        grp_tr = tr_data.groupby("sample_id", sort=False).size().values
        
        fold_oof = np.zeros(va_mask.sum(), dtype="float64")
        for p_idx, params in enumerate(params_list):
            for s in seeds:
                model = lgb.LGBMRanker(**{**params, "random_state": s + 100 * p_idx})
                model.fit(tr_data[feats], tr_data["target_index"], group=grp_tr, categorical_feature=["task_index"])
                
                pred_va = model.predict(va_data[feats])
                s_va = pd.Series(pred_va, index=va_data.index)
                z_va = (s_va - s_va.groupby(va_data["sample_id"]).transform("mean")) / (s_va.groupby(va_data["sample_id"]).transform("std") + 1e-6)
                fold_oof += z_va.values / total_models
                
        oof_preds[va_mask] = fold_oof
        h_fold = hits(va_data[["sample_id", "row_index"]].assign(score=fold_oof)).mean()
        print(f"  Fold {f}: top-3 = {h_fold:.4f} ({time.time() - t_f:.0f}s)")

    grp_full = lab.groupby("sample_id", sort=False).size().values
    for p_idx, params in enumerate(params_list):
        for s in seeds:
            model = lgb.LGBMRanker(**{**params, "random_state": s + 100 * p_idx})
            model.fit(lab[feats], lab["target_index"], group=grp_full, categorical_feature=["task_index"])
            pred_te = model.predict(test[feats])
            s_te = pd.Series(pred_te, index=test.index)
            z_te = (s_te - s_te.groupby(test["sample_id"]).transform("mean")) / (s_te.groupby(test["sample_id"]).transform("std") + 1e-6)
            test_preds += z_te.values / total_models

    return oof_preds, test_preds

print("Entrenando ensamble ultra con LightGBM...")
oof_lgb_ultra, test_lgb_ultra = train_oof_and_test(lab_df, test_df, ALL_FEATS, [PARAMS_A, PARAMS_B], seeds=(0, 1, 2))
report(lab_df[["sample_id", "row_index"]].assign(score=oof_lgb_ultra), "lgb_future_ultra (sin post-proceso)")

Base cargada: 541,097 filas | 146 features base.
Calculando nuevas features uf_*...
¡Listo en 21.8s! -> Total features combinadas: 187 (41 nuevas).
Entrenando ensamble ultra con LightGBM...
  Fold 0: top-3 = 0.9513 (296s)
  Fold 1: top-3 = 0.8966 (307s)
  Fold 2: top-3 = 0.8875 (258s)
  Fold 3: top-3 = 0.8951 (378s)
  Fold 4: top-3 = 0.9075 (2007s)
lgb_future_ultra (sin post-proceso) CV top3 = 0.9070  (13175 muestras)  min atco 0.754  media por atco 0.897
  por fold: 0:0.951  1:0.897  2:0.888  3:0.895  4:0.908
  por atco: 1:0.754  2:0.881  3:0.930  4:0.874  5:0.902  6:0.890  7:0.938  8:0.841  9:0.940  10:0.811  11:0.952  14:0.967  15:0.929  16:0.856  17:0.896  18:0.949  19:0.928  20:0.900


np.float64(0.9070208728652751)

In [13]:
def softmax_by_sample(df_sub, score_arr, temp=1.0):
    s = np.asarray(score_arr, dtype="float64") / temp
    s_max = pd.Series(s, index=df_sub.index).groupby(df_sub["sample_id"]).transform("max").values
    ex = np.exp(s - s_max)
    denom = pd.Series(ex, index=df_sub.index).groupby(df_sub["sample_id"]).transform("sum").values
    return ex / (denom + 1e-12)

# Cargamos nn_future si está disponible para hacer el ensemble selectivo por tipo
nn_oof_path = ROOT / "futuro" / "oof" / "nn_future.csv"
nn_preds_path = ROOT / "futuro" / "preds" / "nn_future.csv"

has_nn = nn_oof_path.exists() and nn_preds_path.exists()
if has_nn:
    nn_oof = lab_df[["sample_id", "row_index"]].merge(pd.read_csv(nn_oof_path), on=["sample_id", "row_index"], how="left")["score"].values
    nn_te = test_df[["sample_id", "row_index"]].merge(pd.read_csv(nn_preds_path), on=["sample_id", "row_index"], how="left")["score"].values
    p_nn_oof = softmax_by_sample(lab_df, nn_oof)
    p_nn_te = softmax_by_sample(test_df, nn_te)
    print("nn_future cargado correctamente para blending condicional.")

p_lgb_oof = softmax_by_sample(lab_df, oof_lgb_ultra)
p_lgb_te = softmax_by_sample(test_df, test_lgb_ultra)

# Tipos donde la red neuronal es competitiva o complementaria (entry_coord=2, exit_coord=3, exit_cond=10, rtr=13)
COORD_MASK_OOF = lab_df["task_index"].isin([2, 3, 10, 13]).values
COORD_MASK_TE = test_df["task_index"].isin([2, 3, 10, 13]).values

def blend_scores(p_lgb, p_nn, coord_mask, w_global=0.08, w_coord=0.25):
    if p_nn is None:
        return p_lgb
    w = np.where(coord_mask, w_coord, w_global)
    p_mix = (1.0 - w) * p_lgb + w * p_nn
    return p_mix

def temporal_smooth_with_aircraft_inertia(df_meta, prob_arr, k=4, decay=0.80, ac_bonus=0.12):
    """
    Suaviza la probabilidad de cada tarea con los ±k snapshots contiguos del mismo bloque (block),
    y añade un pequeño bono si el MISMO AVIÓN (cs) tenía probabilidad alta en t-1 o t+1
    (captura transiciones inmediatas entry_coord <-> assume y exit_coord <-> transfer).
    """
    x = df_meta[["sample_id", "row_index", "task_name", "cs", "block", "blk_pos"]].copy()
    x["p"] = prob_arr
    
    # 1. Suavizado por tarea exacta (block, task_name)
    x = x.sort_values(["block", "task_name", "blk_pos"])
    g_task = x.groupby(["block", "task_name"], sort=False)["p"]
    num = x["p"].copy()
    den = pd.Series(1.0, index=x.index)
    
    for i in range(1, k + 1):
        w = decay ** i
        for sh in (g_task.shift(i), g_task.shift(-i)):
            num += w * sh.fillna(0.0)
            den += w * sh.notna().astype("float64")
            
    x["p_smooth"] = num / den
    
    # 2. Inercia de avión en ±1 snapshot dentro del mismo bloque
    if ac_bonus > 0:
        ac_snap = x.groupby(["block", "cs", "blk_pos"], sort=False)["p"].max().reset_index()
        ac_snap = ac_snap.sort_values(["block", "cs", "blk_pos"])
        g_ac = ac_snap.groupby(["block", "cs"], sort=False)["p"]
        ac_snap["ac_Context"] = 0.5 * (g_ac.shift(1).fillna(0.0) + g_ac.shift(-1).fillna(0.0))
        x = x.merge(ac_snap[["block", "cs", "blk_pos", "ac_Context"]], on=["block", "cs", "blk_pos"], how="left")
        x["p_smooth"] = x["p_smooth"] + ac_bonus * x["ac_Context"]
        
    x = x.sort_values(["sample_id", "row_index"])
    return x["p_smooth"].values

# Preparar bloques contiguos si no están en lab_df / test_df
blk_all = ff.add_blocks(ff.add_folds(ff.load()))[["sample_id", "row_index", "block", "blk_pos"]]
if "block" not in lab_df.columns:
    lab_df = lab_df.merge(blk_all, on=["sample_id", "row_index"], how="left")
    test_df = test_df.merge(blk_all, on=["sample_id", "row_index"], how="left")

# Búsqueda en CV anidada de los hiperparámetros de post-procesado
grid_params = list(itertools.product(
    [0.0, 0.08] if has_nn else [0.0],   # w_global
    [0.0, 0.22] if has_nn else [0.0],   # w_coord
    [3, 5],                             # k vecinos
    [0.75, 0.85],                       # decay
    [0.0, 0.10]                         # ac_bonus (inercia avión)
))

print("Evaluando post-procesamiento con CV anidada (5 folds)...")
precomputed = {}
for gp in grid_params:
    wg, wc, k, dec, acb = gp
    p_mix = blend_scores(p_lgb_oof, p_nn_oof if has_nn else None, COORD_MASK_OOF, wg, wc)
    precomputed[gp] = temporal_smooth_with_aircraft_inertia(lab_df, p_mix, k=k, decay=dec, ac_bonus=acb)

oof_final_nested = np.zeros(len(lab_df), dtype="float64")
for f in range(5):
    tr_m = (lab_df["fold"] != f).values
    va_m = (lab_df["fold"] == f).values
    best_gp = max(grid_params, key=lambda gp: hits(lab_df.loc[tr_m, ["sample_id", "row_index"]].assign(score=precomputed[gp][tr_m])).mean())
    oof_final_nested[va_m] = precomputed[best_gp][va_m]
    h_va = hits(lab_df.loc[va_m, ["sample_id", "row_index"]].assign(score=oof_final_nested[va_m])).mean()
    print(f"  Fold {f} (validación honesta): top-3 = {h_va:.4f} | params elegidos en train: {best_gp}")

report(lab_df[["sample_id", "row_index"]].assign(score=oof_final_nested), "lgb_future_ultra + post-proceso (CV Anidada)")

# Elegimos el mejor conjunto global para generar las predicciones de test y guardar
best_global_gp = max(grid_params, key=lambda gp: hits(lab_df[["sample_id", "row_index"]].assign(score=precomputed[gp])).mean())
wg, wc, k, dec, acb = best_global_gp
print(f"\nParámetros óptimos globales para test: w_global={wg}, w_coord={wc}, k={k}, decay={dec}, ac_bonus={acb}")

oof_best = precomputed[best_global_gp]
p_mix_te = blend_scores(p_lgb_te, p_nn_te if has_nn else None, COORD_MASK_TE, wg, wc)
test_best = temporal_smooth_with_aircraft_inertia(test_df, p_mix_te, k=k, decay=dec, ac_bonus=acb)

# Guardar en el formato estándar de la rama futuro/
OUT_NAME = "lgb_future_ultra"
for folder in ("oof", "preds", "submissions"):
    (ROOT / "futuro" / folder).mkdir(parents=True, exist_ok=True)

lab_df[["sample_id", "row_index"]].assign(score=oof_best).to_csv(ROOT / "futuro" / "oof" / f"{OUT_NAME}.csv", index=False)
pred_df = test_df[["sample_id", "row_index"]].assign(score=test_best)
pred_df.to_csv(ROOT / "futuro" / "preds" / f"{OUT_NAME}.csv", index=False)

sub = (
    pred_df.sort_values(["sample_id", "score"], ascending=[True, False])
    .groupby("sample_id")["row_index"]
    .apply(lambda r: " ".join(map(str, r.iloc[:3])))
    .rename("prediction")
    .reset_index()
)
sub.to_csv(ROOT / "futuro" / "submissions" / f"sub_{OUT_NAME}.csv", index=False)
print(f"Guardado en futuro/oof/{OUT_NAME}.csv, futuro/preds/{OUT_NAME}.csv y futuro/submissions/sub_{OUT_NAME}.csv ({len(sub)} muestras)")

nn_future cargado correctamente para blending condicional.
Evaluando post-procesamiento con CV anidada (5 folds)...
  Fold 0 (validación honesta): top-3 = 0.9537 | params elegidos en train: (0.0, 0.22, 5, 0.75, 0.0)
  Fold 1 (validación honesta): top-3 = 0.8941 | params elegidos en train: (0.08, 0.0, 5, 0.75, 0.0)
  Fold 2 (validación honesta): top-3 = 0.8919 | params elegidos en train: (0.08, 0.0, 3, 0.75, 0.0)
  Fold 3 (validación honesta): top-3 = 0.8983 | params elegidos en train: (0.0, 0.22, 5, 0.75, 0.0)
  Fold 4 (validación honesta): top-3 = 0.9024 | params elegidos en train: (0.0, 0.22, 5, 0.85, 0.0)
lgb_future_ultra + post-proceso (CV Anidada) CV top3 = 0.9074  (13175 muestras)  min atco 0.746  media por atco 0.897
  por fold: 0:0.954  1:0.894  2:0.892  3:0.898  4:0.902
  por atco: 1:0.746  2:0.884  3:0.919  4:0.877  5:0.893  6:0.901  7:0.946  8:0.858  9:0.947  10:0.814  11:0.951  14:0.974  15:0.930  16:0.856  17:0.896  18:0.948  19:0.927  20:0.887

Parámetros óptimos globales